In [1]:
# -*- coding: utf-8 -*-
"""
Add Adopters (True/False) to the main dataset.

RQ1 definition:
  Adopters := (AT == 1) AND (B == 1 OR Y == 1)
  where:
    AT = Instru_test
    B  = build_instru_signal
    Y  = ci_android_signal

Reads from:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet

Outputs:
  Overwrites the original input file (CSV/XLSX).
"""

from pathlib import Path
import pandas as pd
import re

# ------------------- CONFIG -------------------
DATA_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
# Prefer this file name if present; otherwise the script will pick the first CSV/XLSX it finds:
PREFERRED_FILE = "5.0_Total_Repo.csv"

# Column names (change here if your dataset uses different names)
COL_AT = "Instru_test"
COL_B  = "instru_t_signal_config"
COL_Y  = "ci_android_signal"

# New column name to add:
RQ1_COL = "Adopters"
# ----------------------------------------------


def find_input_file(dir_path: str, preferred: str = None) -> Path:
    p = Path(dir_path)
    if preferred and (p / preferred).exists():
        return p / preferred
    # Fallbacks: first CSV then first Excel
    for pattern in ("*.csv", "*.xlsx", "*.xls"):
        files = sorted(p.glob(pattern))
        if files:
            return files[0]
    raise FileNotFoundError(f"No CSV/XLSX file found in {dir_path!r}.")


def robust_to01(series: pd.Series) -> pd.Series:
    """Coerce booleans/strings/numbers like '1', '1.0', 'true', 'yes' -> 1; else 0."""
    s = series.copy()

    # Already boolean?
    if s.dtype == bool:
        return s.astype(int)

    # Numeric? threshold at > 0
    if pd.api.types.is_numeric_dtype(s):
        return (s.fillna(0).astype(float) > 0).astype(int)

    # String-like: normalize tokens
    t = s.astype(str).str.strip().str.lower()
    token_map = {
        "1":1, "true":1, "yes":1, "y":1, "t":1, "on":1, "present":1,
        "0":0, "false":0, "no":0, "n":0, "f":0, "off":0, "":0, "none":0, "null":0, "na":0, "nan":0
    }
    mapped = t.map(token_map)

    # Numeric-looking strings like "1.0", "0.0", "+2", etc.
    numeric_like = pd.to_numeric(t.str.replace(r"[^0-9\.\-]+", "", regex=True), errors="coerce")
    num01 = (numeric_like.fillna(0).astype(float) > 0).astype(int)

    out = mapped.where(mapped.notna(), num01).fillna(0).astype(int)
    return out


def main():
    in_path = find_input_file(DATA_DIR, PREFERRED_FILE)
    print(f"Reading: {in_path}")

    # Load file
    if in_path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(in_path)
        filetype = "excel"
    else:
        df = pd.read_csv(in_path, encoding="utf-8-sig")
        filetype = "csv"

    # Sanity check columns exist (with graceful message)
    missing = [c for c in (COL_AT, COL_B, COL_Y) if c not in df.columns]
    if missing:
        print("\n⚠️  The following expected columns were not found exactly as named:")
        for c in missing:
            print(f"   - {c}")
        print("\nAvailable columns are:")
        print(list(df.columns))
        print("\nIf your dataset uses slightly different names, edit COL_AT/COL_B/COL_Y at the top and re-run.")
        # Attempt a soft match by normalized names to help recover from minor header differences
        def clean_name(name: str) -> str:
            n = str(name).strip().lower()
            n = re.sub(r"[\s\-\./]+", "_", n)
            n = re.sub(r"[^a-z0-9_]+", "", n)
            n = re.sub(r"_+", "_", n).strip("_")
            return n
        norm_lookup = {clean_name(c): c for c in df.columns}
        wanted = {COL_AT: None, COL_B: None, COL_Y: None}
        for logical, target in [(COL_AT, "instru_test"), (COL_B, "build_instru_signal"), (COL_Y, "ci_android_signal")]:
            key = clean_name(logical)
            if key in norm_lookup:
                wanted[logical] = norm_lookup[key]
        if any(v is None for v in wanted.values()):
            print("\n🔎 Could not safely map all three columns automatically—aborting to avoid wrong results.")
            return
        else:
            print("\n✅ Soft-mapped columns by normalized names:")
            for k,v in wanted.items():
                print(f"   {k} -> {v}")
            at_col, b_col, y_col = wanted[COL_AT], wanted[COL_B], wanted[COL_Y]
    else:
        at_col, b_col, y_col = COL_AT, COL_B, COL_Y

    # Normalize to 0/1
    AT = robust_to01(df[at_col])
    B  = robust_to01(df[b_col])
    Y  = robust_to01(df[y_col])

    # RQ1 flag (boolean)
    rq1_flag = (AT.eq(1) & (B.eq(1) | Y.eq(1)))
    df[RQ1_COL] = rq1_flag  # True/False

    # Optional: also add a 0/1 version if you like
    df[RQ1_COL + "_int"] = rq1_flag.astype(int)

    # Quick summary
    total = len(df)
    adopters = int(rq1_flag.sum())
    rate = adopters / total * 100 if total else 0.0
    print(f"\nAdded column: {RQ1_COL}")
    print(f"Adopters by RQ1: {adopters} of {total}  ({rate:.1f}%)")

    # --------- SAVE: OVERWRITE THE INPUT FILE ----------
    if filetype == "excel":
        # Overwrite with a single sheet named 'data' (same behavior as before)
        with pd.ExcelWriter(in_path, engine="xlsxwriter") as xw:
            df.to_excel(xw, index=False, sheet_name="data")
    else:
        # Keep UTF-8 with BOM to mirror the read encoding
        df.to_csv(in_path, index=False, encoding="utf-8-sig")

    print(f"[OK] Overwrote: {in_path}")

if __name__ == "__main__":
    main()


Reading: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\5.0_Total_Repo.csv

Added column: Adopters
Adopters by RQ1: 1548 of 4518  (34.3%)
[OK] Overwrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\5.0_Total_Repo.csv
